# Détection des Glissements de Terrain par IA et Topographie

## Introduction
Les glissements de terrain sont des menaces majeures en RDC, particulièrement dans les zones montagneuses de l'Est. Ce notebook combine l'imagerie satellite et les données d'élévation pour identifier les zones de ruptures récentes et évaluer la susceptibilité du terrain.

## Objectifs
*   **Identification des cicatrices** : Repérer les zones de dénudation brutale du sol sur des pentes fortes.
*   **Analyse de pente** : Utiliser les données SRTM pour isoler les terrains instables.
*   **Cartographie d'urgence** : Générer des alertes spatialisées exploitables pour les secours.

## Méthodologie
1.  **Setup** : Installation des outils d'analyse géospatiale.
2.  **Acquisition** : Imagerie Sentinel-2 et Modèle Numérique de Terrain SRTM.
3.  **Analyse de Risque** : Algorithme croisant la pente (>20°) et la baisse de l'indice végétal.
4.  **Visualisation** : Carte des zones de glissements potentiels.

In [ ]:
# ====================================================
# ÉTAPE 1 : Configuration
# ====================================================
!pip install geemap earthengine-api rasterio opencv-python matplotlib -q

import ee, geemap, rasterio, cv2
import numpy as np
import matplotlib.pyplot as plt

try: ee.Initialize()
except: 
    ee.Authenticate()
    ee.Initialize(project='geocongoai-api')

print('✅ Système prêt')

## Zone d'Étude (ROI)
Ciblage de la zone d'étude régionale.

In [ ]:
# ====================================================
# ÉTAPE 2 : Définition de la ROI
# ====================================================
roi = ee.Geometry.Rectangle([15.0, -5.0, 16.0, -4.0])

Map = geemap.Map(basemap='Esri.WorldImagery')
Map.centerObject(roi, 8)
Map.addLayer(roi, {'color': 'red'}, 'Zone d'étude')
Map

## Acquisition Multi-Source
Extraction conjointe du signal optique (S2) et du relief (SRTM).

In [ ]:
# ====================================================
# ÉTAPE 3 : Données Satellite et Relief
# ====================================================
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(roi).filterDate('2023-01-01', '2023-12-31').median().clip(roi)
dem = ee.Image('USGS/SRTMGL1_003').clip(roi)
slope = ee.Terrain.slope(dem)

geemap.ee_export_image(s2.select(['B4', 'B8']).addBands(slope), 'landslide_input.tif', scale=30, region=roi)

## Calcul de la Susceptibilité
Un glissement est ici modélisé par une zone de faible végétation (NDVI bas) ayant une forte rugosité de texture sur une pente importante.

In [ ]:
# ====================================================
# ÉTAPE 4 : Algorithme de Détection
# ====================================================
with rasterio.open('landslide_input.tif') as src: data = src.read().astype(np.float32)
red, nir, slope_val = data[0], data[1], data[2]
ndvi = (nir - red) / (nir + red + 1e-8)

# Détection de contours de texture
edges = cv2.Canny(red.astype(np.uint8), 30, 100) / 255.0

# Formule de susceptibilité croisée
susceptibility = (1.0 - ndvi) * edges * (slope_val / 90.0)

plt.figure(figsize=(10, 8))
plt.imshow(susceptibility, cmap='YlOrRd')
plt.colorbar(label='Indice de risque de glissement')
plt.title('Carte des Zones à Risque de Glissements de Terrain')
plt.axis('off')
plt.show()